In [17]:
from dotenv import load_dotenv
import os
import ollama
import json
from googleapiclient.discovery import build


## Load environment variables from .env

In [18]:
load_dotenv("../.env")

OLLAMA_CF_CLIENT_ID = os.getenv("OLLAMA_CF_CLIENT_ID")
OLLAMA_CF_CLIENT_SECRET = os.getenv("OLLAMA_CF_CLIENT_SECRET")
OLLAMA_HOST = os.getenv("OLLAMA_HOST")

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

# Model used
MODEL = "qwen3.5:9b"

##  Configuration of Ollama client through cloudflare 

In [3]:
cf_headers = {
    "CF-Access-Client-Id": os.environ.get("OLLAMA_CF_CLIENT_ID"),
    "CF-Access-Client-Secret": os.environ.get("OLLAMA_CF_CLIENT_SECRET"),
}

host = os.environ.get("OLLAMA_HOST")

client = ollama.Client(host=host, headers=cf_headers)

In [4]:
# Test prompt to verify
response = client.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say Hi to me and ask how my day was"}
    ]
)

print("Ollama connection test\n")
print(response.message.content)

Ollama connection test

Hi there! 👋 How was your day?


## Mock Data for testing

In [5]:
mock_input = {
    "user_id": "123",
    "sentiment_label": "Anxiety",
    "mood_trend": "improving",
    "selected_tags": ["Calm & Peace", "Music"] 
}

## Prompt Engineering and query generation

In [ ]:
def build_search_query(mock_input):
    sentiment = mock_input["sentiment_label"]
    mood_trend = mock_input["mood_trend"]
    tags = ", ".join(mock_input["selected_tags"])  # ["Calm & Peace", "Music"] to "Calm & Peace, Music"

    
    prompt = f"""

    You are a mental wellness YouTube video recommendation assistant.
Your sole purpose is to find videos that are safe, appropriate, and genuinely 
helpful for someone based on their current emotional state and preferences.

Current mood: {sentiment}
Mood trend: {mood_trend}
User preferred video types: {tags}

Mood Rules

Suicidal:
    Only recommend extremely gentle and grounding content.
    Every query must feel safe, warm, and non-triggering.
    Never suggest anything emotionally intense, heavy, or overwhelming.
    Themes: "you are not alone", calm nature visuals, soft reassuring voices.

Depression:
    Recommend soft, comforting, and quietly hopeful content.
    Avoid high energy content and toxic positivity at all costs.
    Themes: cozy vlogs, soft spoken advice, personal healing journeys.

Anxiety:
    Recommend calming content that slows the mind down.
    Target nervous system regulation, breathing, and present moment awareness.
    Avoid anything fast paced, loud, or overstimulating.
    Themes: breathwork, guided meditation, anxiety education, calm music.

Stress:
    Recommend light and gently relieving content.
    Avoid complex, heavy, or problem solving content.
    Themes: satisfying videos, light documentaries, stress relief techniques.

Normal:
    Recommend engaging, motivational, and growth oriented content.
    Avoid emotionally heavy or overly therapeutic content.
    Themes: productivity, interesting science, motivational talks.

Trend Rules

Improving:
    Lean slightly more positive and uplifting across all queries.
    Acceptable to include mildly energizing or inspiring content.

Declining:
    Apply extra gentleness and emotional care to every query.
    Prioritize emotional safety above everything else including engagement.

Tag Rules

The user has selected their preferred video types: {tags}
These preferences must be reflected across the generated queries.

Calm and Peace: meditation, nature sounds, ambient music, sleep content.
Educational: psychology explainers, TED style talks, mind related science.
Documentary: short human interest documentaries, real life personal journeys.
Science: neuroscience, mind body connection, biology of emotions.
Music: lofi playlists, instrumental music, mood based music.

Always blend the mood rules and tag preferences naturally into each query.
Never generate content that could lead to harmful or triggering results
for someone in a vulnerable emotional state. When in doubt, choose the gentler option.

Output

Generate exactly 5 YouTube search queries based on everything above.
Each query must be a maximum of 8 words and sound like something a real person would search.
Return only a JSON array of 5 strings.
Example: ["query one", "query two", "query three", "query four", "query five"]
No explanation. No extra text. No markdown. Just the JSON array.
"""
    # Send prompt for query generation
    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.message.content.strip()

    queries = json.loads(raw)
    return queries

queries = build_search_query(mock_input)
print("Generated search queries \n")
for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")

Generated search queries 

1. Soft lofi music for anxiety relief and calm
2. Guided breathing exercises to calm anxiety and relax
3. Gentle nature sounds for peaceful anxiety and sleep
4. Uplifting calm playlist to soothe anxious mind now
5. Calm meditation music to ease anxiety stress today


# YouTube Video Fetcher

In [ ]:
# fetching YouTube videos based on a list of search queries provided above
def fetch_youtube_videos(queries, max_per_query=2):
    youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

    all_videos = [] 
    seen_ids = set() # track video ids and avoid duplicate

    for query in queries:
        # Searching youtube videos for each generated query
        try:
            request = youtube.search().list(
                q=query, 
                part="snippet",
                type="video",
                maxResults=max_per_query,
                safeSearch="strict",
                relevanceLanguage="en",
                videoDuration="medium"
            )
            response = request.execute()

            for item in response.get("items", []):
                video_id = item["id"]["videoId"]

                if video_id in seen_ids:
                    continue

                seen_ids.add(video_id)
                snippet = item["snippet"]

                video = {
                    "video_id": video_id,
                    "title": snippet["title"],
                    "channel": snippet["channelTitle"],
                    "thumbnail": snippet["thumbnails"]["high"]["url"],
                    "url": "https://www.youtube.com/watch?v=" + video_id,
                    "query_used": query 
                }

                all_videos.append(video)

        except Exception as e:
            print(f"Error fetching videos for query: {query}")
            print(e)

    return all_videos

videos = fetch_youtube_videos(queries, max_per_query=2)

print("Fetched videos:")
for i, video in enumerate(videos, 1):
    print(f"{i}. {video['title']}")
    print(f"{video['url']}\n")

Fetched videos:
1. songs to help you calm your anxiety  [ lofi | jazzhop | chill mix ]
https://www.youtube.com/watch?v=JspBhPzs7i8

2. Fresh Mind Relaxing 💗 | Slowed + Reverb Lofi Mix for Calm, Focus &amp; Peaceful Vibes #sleepmusic
https://www.youtube.com/watch?v=41GGR0C1jaI

3. 4-7-8 Calm Breathing Exercise | 10 Minutes of Deep Relaxation | Anxiety Relief | Pranayama Exercise
https://www.youtube.com/watch?v=LiUnFJ8P4gM

4. 4-7-8 Calm Breathing Exercise | 15 Minutes of Deep Relaxation | Anxiety Relief | Pranayama Exercise
https://www.youtube.com/watch?v=lEzaFx8k7Ew

5. 15 Minute Meditation Music, Calm Music, Relax, Meditation, Stress Relief, Spa, Study, Sleep, ☯3527B
https://www.youtube.com/watch?v=AImuCtIokl0

6. 5 Minute Music Meditation | 5 Minutes to Calm | Scenic Waterfall and Nature Sounds
https://www.youtube.com/watch?v=ETdAlFuJ9PE

7. Positive Affirmations for Peace and Calm | Reduce Stress &amp; Anxiety
https://www.youtube.com/watch?v=oS6KlpzDNS0

8. So, You&#39;re Having an 